# Smart Budget — SageMaker Endpoint

**Ticket:** DATA-1140 · **Endpoint:** `smart-budget-suggestion-endpoint`

> **Kernel:** usar el mismo kernel donde funciona el proyecto SAFE
> (`sagemaker` SDK clásico con `SKLearnModel` y `get_execution_role`).

| Step | Qué hace |
|------|---------|
| 1 | Setup — sesión y rol |
| 2 | Empaquetar y subir `model.tar.gz` |
| 3 | Crear y desplegar endpoint (`SKLearnModel`) |
| 4 | Probar endpoint |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPATIBILIDAD: sagemaker SDK
#
# sagemaker 4.x cambió la arquitectura interna — requiere sagemaker_core como
# módulo Python y rompe SKLearnModel / get_execution_role en Studio.
# sagemaker 3.8.5 es la última versión estable compatible con este entorno.
#
# Ejecuta esta celda UNA VEZ → reinicia el kernel → continúa desde Cell 2.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "sagemaker==3.8.5", "--quiet", "--force-reinstall"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ sagemaker 3.8.5 instalado")
    print("⚠️  REINICIA EL KERNEL ahora: Kernel → Restart Kernel")
else:
    print(f"❌ Error:\n{result.stderr[-500:]}")


In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role, Session
import sagemaker
import boto3, json, os, shutil, tarfile, time
from pathlib import Path

# Setup — igual que SAFE
sagemaker_session = Session()
role = get_execution_role()

S3_BUCKET     = 'blossom-analytics-safe-dev-nv'
S3_KEY        = 'smart_budget/endpoint/v1/model.tar.gz'
S3_URI        = f's3://{S3_BUCKET}/{S3_KEY}'
ENDPOINT_NAME = 'smart-budget-suggestion-endpoint'

print(f"Role    : {role}")
print(f"Region  : {sagemaker_session.boto_region_name}")
print(f"S3 URI  : {S3_URI}")


---
## Step 2 — Empaquetar `model.tar.gz` y subir a S3

In [ ]:
REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())

SRC_SMART_BUDGET = REPO_ROOT / 'src' / 'smart_budget'
DATA_DIR         = REPO_ROOT / 'data' / 'dough'
ARTIFACTS_DIR    = REPO_ROOT / 'notebooks' / 'model_artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Staging
staging = ARTIFACTS_DIR / 'staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

# smart_budget package (inference.py lo lee desde aquí)
shutil.copytree(SRC_SMART_BUDGET, staging / 'smart_budget')

# CSVs de datos
data_staging = staging / 'data'
data_staging.mkdir()
for name in ['smart_budget_synthetic.csv']:
    src = DATA_DIR / name
    if src.exists(): shutil.copy(src, data_staging / name)
    else: print(f"⚠️  No encontrado: {src}")
for name in ['test_internal.csv', 'test_external.csv']:
    src = DATA_DIR / 'test' / name
    if src.exists(): shutil.copy(src, data_staging / name)
    else: print(f"⚠️  No encontrado: {src}")

# Tarball
tarball_path = ARTIFACTS_DIR / 'model.tar.gz'
with tarfile.open(tarball_path, 'w:gz') as tar:
    for item in staging.rglob('*'):
        if item.is_file():
            tar.add(item, arcname=item.relative_to(staging))

print(f"✅ model.tar.gz  ({tarball_path.stat().st_size / 1024:.1f} KB)")

# Upload a S3
s3 = sagemaker_session.boto_session.client('s3')
s3.upload_file(str(tarball_path), S3_BUCKET, S3_KEY)
print(f"✅ Subido: {S3_URI}")


---
## Step 3 — Crear y desplegar endpoint

In [ ]:
# src/sagemaker/ contiene SOLO inference.py — sin FastAPI ni router.py
# model.tar.gz contiene smart_budget package + CSVs (cargados por model_fn)
# SageMaker maneja el HTTP layer internamente (gunicorn+Flask propios del container)
sk_model = SKLearnModel(
    model_data=S3_URI,
    role=role,
    entry_point='inference.py',
    source_dir=str(REPO_ROOT / 'src' / 'sagemaker'),
    framework_version='1.2-1',
    sagemaker_session=sagemaker_session,
)

predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=ENDPOINT_NAME,
)

print(f"✅ Endpoint desplegado: {ENDPOINT_NAME}")


---
## Step 4 — Probar el endpoint

In [ ]:
runtime = sagemaker_session.boto_session.client('sagemaker-runtime')

def invoke(payload):
    r = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Body=json.dumps(payload),
    )
    return json.loads(r['Body'].read())

# Happy path
result = invoke({'idaccount': 'EXT2', 'defaultcategory': 'Food & Dining', 'period_id': '2026-05'})
print(json.dumps(result, indent=2))


In [ ]:
import botocore

# Regla 1 — Cuenta no existe → error
try:
    invoke({'idaccount': 'CUENTA_INEXISTENTE', 'defaultcategory': 'Groceries', 'period_id': '2026-05'})
    print("❌ Regla 1 FALLÓ")
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print("✅ Regla 1 — cuenta inexistente → error")

# Regla 2 — Categoría inválida → error
try:
    invoke({'idaccount': 'EXT2', 'defaultcategory': 'CategoriaFalsa', 'period_id': '2026-05'})
    print("❌ Regla 2 FALLÓ")
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print("✅ Regla 2 — categoría inválida → error")

# Regla 3 — Sin datos → null
r = invoke({'idaccount': 'SYN001', 'defaultcategory': 'Groceries', 'period_id': '2026-05'})
assert r['suggested_amount'] is None
print(f"✅ Regla 3 — sin datos → null  ({r.get('display_label')})")


---
## ⚠️ Borrar endpoint cuando termines

Genera costo por hora.

In [ ]:
# --- Delete EndpointConfig ---
client = boto3.client("sagemaker")
endpoint_config_name = ENDPOINT_NAME

try:
    client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"EndpointConfig '{endpoint_config_name}' deleted.")
except client.exceptions.ClientError as e:
    if "Could not find endpoint configuration" in str(e):
        print("EndpointConfig does not exist, continuing.")
    else:
        raise


In [ ]:
# --- Force delete Endpoint ---
client = boto3.client("sagemaker", region_name="us-east-1")
client.delete_endpoint(EndpointName=endpoint_config_name)
# client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
